In [4]:
import sys
import os
sys.path.append(os.path.abspath('..'))

# print("Python path:", sys.path)
# print("Parent folder contents:", os.listdir('..'))
# print("Current folder:", os.getcwd())

import inspect
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from src.lwr_model import point_density, number_of_cells, linear_density_gradient, clean_count_data
from src.lwr_model import FREE_SPEEDS, CELL_SIZE
from src.lwr_model.CTM import CTM_model
from src.lwr_model import distances

print(inspect.getsource(distances))

# --- Real world density ---
df_count = clean_count_data()
density_at_point = point_density(df_count)
cell_lengths = number_of_cells(density_at_point)
whole_density_arr = linear_density_gradient(density_at_point, cell_lengths)

# --- Model setup ---
density_init = np.clip(np.array(whole_density_arr), 0, 0.15 * 0.99)
N = len(density_init)
cell_widths = np.full(N, CELL_SIZE)

v_free = np.mean(list(FREE_SPEEDS.values()))
jam_density = 0.15
max_flow = v_free * jam_density / 4
inlet_density = density_init[0]
simulation_time = 3600

# --- Run ---
model = CTM_model(cell_widths, density_init, jam_density, max_flow, v_free)
model.run(inlet_density=inlet_density, total_time=simulation_time)

# --- Plot ---
history = np.array(model.history)
distance = np.arange(N) * CELL_SIZE

mpl.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
im = axes[0].imshow(
    history,
    aspect='auto',
    origin='lower',
    extent=[0, N * CELL_SIZE, 0, len(history) * model.step_width],
    cmap='plasma'
)
cbar = fig.colorbar(im, ax=axes[0], pad=0.02)
cbar.set_label('Density (veh/m)', fontsize=11)
cbar.outline.set_visible(False)
axes[0].set_xlabel('Distance (m)')
axes[0].set_ylabel('Time (s)')
axes[0].set_title('Density over space and time', fontweight='bold', pad=12)

# Initial vs final
axes[1].plot(distance, history[0], label='Initial', color='steelblue', linewidth=1.5)
axes[1].plot(distance, history[-1], label='Final', color='tomato', linewidth=1.5)
axes[1].axhline(jam_density / 2, color='grey', linestyle='--', linewidth=0.8, label='Critical density')
axes[1].set_xlabel('Distance (m)')
axes[1].set_ylabel('Density (veh/m)')
axes[1].set_title('Initial vs final density', fontweight='bold', pad=12)
axes[1].legend(frameon=False)

plt.tight_layout()
plt.show()

# --- Travel time ---
travel = model.travel_time(start_cell=0, end_cell=N-1, start_time=0)
print(f"Travel time: {travel/60:.2f} min" if travel else "Vehicle did not reach end within simulation")

import requests
import math
import certifi

def distance_lat_long(lat1, lon1, lat2, lon2):
    osrm_url = f"https://router.project-osrm.org/route/v1/driving/{lon1},{lat1};{lon2},{lat2}"
    params = {"overview": "full", "geometries": "geojson"}
    osrm_req = requests.get(osrm_url, params=params, verify=certifi.where())
    osrm_data = osrm_req.json()
    route = osrm_data['routes'][0]
    distance = route['distance']
    return distance


def walk_lat_long(lat1, lon1, lat2, lon2):
    osrm_url = f"https://router.project-osrm.org/route/v1/walking/{lon1},{lat1};{lon2},{lat2}"
    params = {"overview": "full", "geometries": "geojson"}
    osrm_req = requests.get(osrm_url, params=params, verify=certifi.where())
    osrm_data = osrm_req.json()
    route = osrm_data['routes'][0]
    distance = route['distance']
    time = route['duration']
    geometries = route['geometry']['coordinates']
    return distance, time, geometries

def haversine_distance(lat1, lon1, lat2, lon2):
    phi1, phi2 =

AttributeError: 'CTM_model' object has no attribute 'run'